In [1]:
#!/usr/bin/env python3

import numpy as np
import pandas as pd
import multiprocessing as mp
import h5py
import tqdm
import os

from tdinf import run_sampler, group_postprocess

# ============================================================
# SETTINGS YOU CHANGE
# ============================================================

# directory containing your full run
RUN_DIR = "/home/kgk3/tdinf/examples/GW190521/output/full_0.0seconds"

# posterior samples
POSTERIOR_FILE = f"{RUN_DIR}/full_0.0seconds.dat"

# command line file used for the full run
COMMAND_FILE = f"{RUN_DIR}/command_line.sh"

# output waveform file
OUTPUT_FILE = f"{RUN_DIR}/full_waveforms.h5"

# number of posterior waveforms
N_WAVEFORMS = 1000

# cpus
NCPU = 32

# ============================================================
# LOAD LIKELIHOOD MANAGER
# ============================================================

parser = run_sampler.create_run_sampler_arg_parser()

args, kwargs, likelihood_manager = (
    group_postprocess.get_settings_from_command_line_file(
        COMMAND_FILE,
        "full_0.0seconds.dat",
        RUN_DIR + "/",
        parser,
        verbose=True
    )
)

# ============================================================
# LOAD POSTERIOR
# ============================================================

df = pd.read_csv(POSTERIOR_FILE, delim_whitespace=True)

# ============================================================
# MAXIMUM LIKELIHOOD WAVEFORM
# ============================================================

imax = np.argmax(df["ln_posterior"] - df["ln_prior"])

maxL_params = df.iloc[imax]

maxL_waveform = likelihood_manager.waveform_manager.get_projected_waveform(
    maxL_params,
    likelihood_manager.ifos,
    time_dict=likelihood_manager.time_dict,
    f22_start=likelihood_manager.f22_start,
    f_ref=likelihood_manager.f_ref,
)

# ============================================================
# RANDOM POSTERIOR DRAWS
# ============================================================

rand_inds = np.random.randint(len(df), size=N_WAVEFORMS)

def generate_waveform(i):

    params = df.iloc[i]

    return likelihood_manager.waveform_manager.get_projected_waveform(
        params,
        likelihood_manager.ifos,
        time_dict=likelihood_manager.time_dict,
        f22_start=likelihood_manager.f22_start,
        f_ref=likelihood_manager.f_ref,
    )

with mp.Pool(processes=NCPU) as pool:

    waveforms = list(
        tqdm.tqdm(
            pool.imap(generate_waveform, rand_inds),
            total=N_WAVEFORMS
        )
    )

# ============================================================
# SAVE TO HDF5
# ============================================================

with h5py.File(OUTPUT_FILE, "w") as f:

    # save times
    grp = f.create_group("times")

    for ifo, arr in likelihood_manager.time_dict.items():
        grp.create_dataset(ifo, data=arr)

    # save maxL waveform
    grp = f.create_group("maxL")

    for ifo, arr in maxL_waveform.items():
        grp.create_dataset(ifo, data=arr)

    # save posterior waveforms
    for i, wf in enumerate(waveforms):

        grp = f.create_group(f"waveform_{i}")

        for ifo, arr in wf.items():
            grp.create_dataset(ifo, data=arr)

print(f"\nSaved waveforms to:\n{OUTPUT_FILE}")

/Users/kylakelley/Documents/Research/tdinf/tdinf/utils/io.py:1: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal
/Users/kylakelley/Documents/Research/tdinf/tdinf/group_postprocess.py:414: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(filename, delimiter='\s+')


FileNotFoundError: [Errno 2] No such file or directory: '/home/kgk3/tdinf/examples/GW190521/output/full_0.0seconds/command_line.sh'